# HiggsLens — Begin Here, From Zero

> **How to use this notebook:** each stage starts with a Markdown explanation. Read the text cell, run the code cell below it, and then interpret the output. The Python cells intentionally avoid explanatory comments; the reasoning belongs in these text boxes.

## The physics in two minutes

The **Higgs field** is a field present throughout space. Interactions with it are associated with the masses of many elementary particles. The **Higgs boson** is a quantum excitation of that field; it was announced by CERN's ATLAS and CMS experiments in 2012. “God particle” is a popular nickname, not a scientific term. The Higgs mechanism does not explain every form of mass: much of the mass of protons and neutrons comes from strong-interaction energy.

## The machine-learning problem

This project does not model the Higgs field or discover a new particle. It studies an established simulated-data task: predict whether each Monte Carlo collision event is labelled **signal (1)** or **background (0)** from 28 numeric features. The first 21 are lower-level kinematic quantities; the final 7 are physicist-derived high-level quantities.

## What we are testing

**Research question:** Under the same data and split budget, how do Logistic Regression, XGBoost and a compact MLP compare when using (1) the 21 low-level features, (2) the 7 high-level features, or (3) all 28 features?

**Product question:** At a declared target signal efficiency, what background rejection and compute cost do candidate models offer? We pick the operating threshold on validation data, then evaluate it unchanged on test data.

## Scientific boundaries

The UCI HIGGS data are simulated, not an unfiltered stream of detector data. The results are a benchmark comparison—not a discovery claim, not a real-world Higgs probability, and not a substitute for physics-specific significance analysis. The assignment recommends beginning with 250,000–1,000,000 rows before trying all 11 million.

**Sources:** [UCI HIGGS dataset](https://archive.ics.uci.edu/dataset/280/higgs) · [CERN Higgs overview](https://home.cern/science/physics/higgs-boson/) · [Baldi et al. (2014)](https://doi.org/10.1038/ncomms5308)


## Read the experiment contract

Before training anything, keep these rules in mind:

- The label column is not an input feature.
- Every model uses the same fixed train/validation/test partitions for this run.
- Scalers learn their parameters from training rows only.
- Validation data selects candidates and the operating threshold. Test data assesses the frozen choices.
- The data source, actual row count, seed, feature set, package versions and compute device are recorded.
- A model is not automatically useful just because it has the highest accuracy; ROC-AUC, average precision, calibration and runtime all matter.


# HiggsLens: Mid-Semester MVP — Simulated Higgs-Event Classification

**Research MVP notebook** | Python, scikit-learn, XGBoost, PyTorch

## Research question
Under a fixed compute and data budget, how do physicist-derived high-level features and a compact neural network compare with classical models for separating simulated Higgs-signal events from background?

## Why this is more than a model leaderboard
The notebook compares low-level, expert-derived and combined feature sets; selects operating thresholds on validation data rather than the test set; measures ranking, probability quality, event-selection trade-offs, and runtime; and exports prediction artifacts for the **HiggsLens** dashboard.

> **Scientific boundary:** UCI HIGGS is simulated Monte Carlo benchmark data. This notebook does not discover the Higgs boson, estimate a real collider discovery significance, or validate a deployable detector system. A physical significance estimate needs event weights, cross sections, luminosity, selections, and uncertainty modeling not supplied by this basic feature table.

### Outputs
- Source and actual row count recorded at runtime.
- Reproducible stratified train/validation/test split for the subset run.
- Logistic Regression, XGBoost (or HistGradientBoosting fallback), and a compact MLP.
- ROC-AUC, average precision, Brier score, fixed-threshold metrics, and threshold performance at a target validation signal efficiency.
- ROC, precision-recall, calibration, feature-importance plots.
- `results.csv`, `val_predictions.csv`, `test_predictions.csv`, `run_metadata.json`, `summary.md`, and a downloadable output ZIP.

**Subset protocol:** this quickstart streams up to 250,000 rows for a fast MVP. Its random split is *not* the original benchmark's official final-500,000 test protocol. The end-semester plan documents the full-data protocol separately.

## 1. Run plan (120-minute MVP)

| Time | Task | Done when |
|---|---|---|
| 0-10 min | Open in Colab; run setup | Imports and runtime check work |
| 10-25 min | Load and inspect data | Actual source, row count, labels and features confirmed |
| 25-65 min | Train locked baseline models | Results table includes every planned model |
| 65-85 min | Inspect curves and explainability | ROC, PR, calibration, importance figures saved |
| 85-105 min | Review exports and write a factual summary | CSVs, metadata and plots are saved |
| 105-120 min | Export bundle; prepare mid-sem demo | ZIP exists; notebook runs top-to-bottom |

Use a GPU if available for the MLP; CPU works but may take longer. If the official stream fails, use the manual official-UCI download path described in the README. The notebook intentionally does not silently substitute a smaller mirror.

## 2. Dataset, task, and feature groups

The UCI HIGGS dataset contains simulated signal/background events. Its original release contains 11 million events, 28 numeric input features, and a binary label. The first 21 inputs are low-level kinematic quantities; the last seven are expert-derived high-level features. UCI lists no missing values for the original dataset.

- **Target:** `1 = signal`, `0 = background`.
- **Low-level features (21):** lepton, missing-energy, and jet measurements.
- **High-level features (7):** `m_jj`, `m_jjj`, `m_lv`, `m_jlv`, `m_bb`, `m_wbb`, `m_wwbb`.

Primary source: [UCI HIGGS dataset](https://archive.ics.uci.edu/dataset/280/higgs), DOI [10.24432/C5V312](https://doi.org/10.24432/C5V312). Background paper: Baldi, Sadowski & Whiteson (2014), [Searching for exotic particles in high-energy physics with deep learning](https://doi.org/10.1038/ncomms5308).

Use the official UCI dataset for the main benchmark so the dataset is aligned to the course brief. The model matrix, sample budget, split protocol and run metadata are part of the result—not incidental details.

## 3. Install/import dependencies

In Colab, most libraries are preinstalled. This cell installs only missing essentials if needed. Restart the runtime only if Colab explicitly asks you to.

In [ ]:
import sys, subprocess, importlib.util

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "torch": "torch",
}
missing = [pip_name for module_name, pip_name in required.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

try:
    import xgboost
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    except Exception as exc:
        print("Optional xgboost install failed; using scikit-learn fallback:", repr(exc))
print("Dependency check complete.")


In [ ]:
import gzip
import io
import os
import time
import json
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    XGBClassifier = None
    HAS_XGBOOST = False
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, brier_score_loss,
    roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
N_EVENTS = 250_000
TEST_SIZE = 0.20
TREE_MAX_ITER = 60
MLP_EPOCHS = 6
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
OUTPUT_DIR = Path("outputs/midsem_250k_seed42")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

FEATURE_NAMES = [
    "lepton_pT", "lepton_eta", "lepton_phi",
    "missing_energy_magnitude", "missing_energy_phi",
    "jet1_pt", "jet1_eta", "jet1_phi", "jet1_btag",
    "jet2_pt", "jet2_eta", "jet2_phi", "jet2_btag",
    "jet3_pt", "jet3_eta", "jet3_phi", "jet3_btag",
    "jet4_pt", "jet4_eta", "jet4_phi", "jet4_btag",
    "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb"
]
LOW_LEVEL = FEATURE_NAMES[:21]
HIGH_LEVEL = FEATURE_NAMES[21:]
UCI_GZIP_URL = "https://mlphysics.ics.uci.edu/data/higgs/HIGGS.csv.gz"


## 4. Load a reproducible subset

The notebook checks `data/raw/HIGGS.csv.gz` first. If that file is absent, it reads the official UCI compressed stream with a row limit, avoiding a full 2.6 GB download. If neither route works, stop and fix the source access; the notebook does not silently substitute a smaller mirror.

In [ ]:
def normalize_binary_target(series):
    s = pd.Series(series).reset_index(drop=True)
    if pd.api.types.is_numeric_dtype(s):
        values = set(pd.to_numeric(s, errors="raise").dropna().unique().tolist())
        if values.issubset({0, 1, 0.0, 1.0}):
            return pd.to_numeric(s).astype(np.int64)
    text = s.astype(str).str.strip().str.lower()
    mapping = {"0": 0, "0.0": 0, "background": 0, "b": 0, "1": 1, "1.0": 1, "signal": 1, "s": 1}
    mapped = text.map(mapping)
    if mapped.isna().any():
        raise ValueError(f"Could not map all target values: {text.unique()[:10]}")
    return mapped.astype(np.int64)


LOCAL_DATA_PATH = Path("data/raw/HIGGS.csv.gz")

def prepare_higgs_frame(frame, source_name):
    if frame.shape[1] != 29:
        raise ValueError(f"Expected 29 columns including the label, got {frame.shape[1]}")
    frame.columns = ["label"] + FEATURE_NAMES
    y = normalize_binary_target(frame.pop("label"))
    return frame, y, source_name


def load_official_subset(nrows):
    if LOCAL_DATA_PATH.exists():
        frame = pd.read_csv(LOCAL_DATA_PATH, header=None, nrows=nrows, compression="gzip", dtype=np.float32)
        return prepare_higgs_frame(frame, f"Official UCI HIGGS local gzip file: {LOCAL_DATA_PATH}")
    request = urllib.request.Request(UCI_GZIP_URL, headers={"User-Agent": "HiggsLens educational benchmark"})
    with urllib.request.urlopen(request, timeout=90) as response:
        with gzip.GzipFile(fileobj=response, mode="rb") as stream:
            frame = pd.read_csv(stream, header=None, nrows=nrows, dtype=np.float32)
    return prepare_higgs_frame(frame, f"Official UCI HIGGS gzip stream (requested cap {nrows:,})")


try:
    X, y, DATA_SOURCE = load_official_subset(N_EVENTS)
except Exception as exc:
    raise RuntimeError(
        "Could not stream the official UCI HIGGS dataset. Open https://archive.ics.uci.edu/dataset/280/higgs, "
        "download the official compressed file, place it at data/raw/HIGGS.csv.gz (or update LOCAL_DATA_PATH), and rerun this cell. "
        "The notebook intentionally does not switch to a smaller dataset without your consent."
    ) from exc

if len(X) != len(y):
    raise ValueError("Feature and target row counts differ.")
if X.shape[1] != 28:
    raise ValueError(f"Expected 28 features; found {X.shape[1]}.")
if not np.isfinite(X.to_numpy(dtype=np.float32)).all():
    raise ValueError("Non-finite values detected; inspect the data before training.")
if set(y.unique()) != {0, 1}:
    raise ValueError(f"Expected both classes 0 and 1; got {sorted(y.unique())}")

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
print(f"Source: {DATA_SOURCE}")
print(f"Rows actually loaded: {len(X):,}")
print(f"Feature count: {X.shape[1]} (21 low-level + 7 high-level)")
print("Label map: 0=background, 1=signal")
print(f"Signal prevalence: {y.mean():.3%}")
display(pd.DataFrame({"Feature group": ["Low-level detector", "Expert-derived high-level", "Total"], "Number of features": [len(LOW_LEVEL), len(HIGH_LEVEL), X.shape[1]]}))


## 5. Quick exploratory data analysis

The label chart checks class balance. The second plot illustrates one expert-derived variable; it is descriptive, not proof that one feature causes the label.

In [ ]:
class_counts = y.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Background (0)", "Signal (1)"], [class_counts.get(0, 0), class_counts.get(1, 0)])
ax.set_title("Class distribution in the loaded subset")
ax.set_ylabel("Number of events")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "class_distribution.png", dpi=160)
plt.show()

plot_n = min(30_000, len(X))
plot_idx = np.random.default_rng(SEED).choice(len(X), size=plot_n, replace=False)
plot_df = X.iloc[plot_idx].copy()
plot_y = y.iloc[plot_idx].to_numpy()
fig, ax = plt.subplots(figsize=(8, 4.5))
for label, name in [(0, "Background"), (1, "Signal")]:
    values = plot_df.loc[plot_y == label, "m_bb"].to_numpy()
    ax.hist(values, bins=45, density=True, alpha=0.5, label=name)
ax.set_title("Example: m_bb distribution by class (sampled events)")
ax.set_xlabel("m_bb")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "m_bb_by_class.png", dpi=160)
plt.show()

print("Missing values by column (top rows):")
display(X.isna().sum().head())
print("Descriptive statistics (first 8 features):")
display(X[FEATURE_NAMES[:8]].describe().T.round(3))


## 6. Split data into train, validation, and untouched test partitions

The 64/16/20 split is for this **subset MVP**, not the original UCI official test protocol. Models fit only on training rows; threshold selection uses validation predictions; the locked test split is used for final reported metrics. Do not tune models after reviewing test outcomes.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, stratify=y_dev, random_state=SEED
)
print(f"Train rows: {len(X_train):,} | Validation rows: {len(X_val):,} | Test rows: {len(X_test):,}")
for split_name, labels in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{split_name.title()} signal rate: {labels.mean():.3%}")

results = []
predictions = {}
predictions_val = {}
predictions_test = {}
model_store = {}
TARGET_SIGNAL_EFFICIENCY = 0.80

def threshold_for_target_recall(y_true, probabilities, target_recall=TARGET_SIGNAL_EFFICIENCY):
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    eligible = np.flatnonzero(tpr >= target_recall)
    if len(eligible) == 0:
        return float(np.min(probabilities))

    best = eligible[np.argmin(fpr[eligible])]
    return float(thresholds[best])

def record_result(name, cols, kind, fitted_model, val_probabilities, test_probabilities,
                  training_seconds, inference_seconds):
    val_probabilities = np.asarray(val_probabilities, dtype=float).reshape(-1)
    test_probabilities = np.asarray(test_probabilities, dtype=float).reshape(-1)
    threshold = threshold_for_target_recall(y_val, val_probabilities)
    test_at_half = (test_probabilities >= 0.5).astype(np.int64)
    test_at_operating = (test_probabilities >= threshold).astype(np.int64)
    row = {
        "Model": name,
        "Feature set": kind,
        "Features": len(cols),
        "Validation_ROC_AUC": roc_auc_score(y_val, val_probabilities),
        "Validation_AP": average_precision_score(y_val, val_probabilities),
        "Test_ROC_AUC": roc_auc_score(y_test, test_probabilities),
        "Test_AP": average_precision_score(y_test, test_probabilities),
        "Test_Accuracy_at_0.5": accuracy_score(y_test, test_at_half),
        "Test_Precision_at_0.5": precision_score(y_test, test_at_half, zero_division=0),
        "Test_Recall_at_0.5": recall_score(y_test, test_at_half, zero_division=0),
        "Test_F1_at_0.5": f1_score(y_test, test_at_half, zero_division=0),
        "Test_Brier": brier_score_loss(y_test, test_probabilities),
        "Validation_threshold_for_80pct_recall": threshold,
        "Test_recall_at_validation_threshold": recall_score(y_test, test_at_operating, zero_division=0),
        "Test_background_rejection_at_validation_threshold": 1.0 - (np.sum((test_at_operating == 1) & (y_test.to_numpy() == 0)) / max(np.sum(y_test.to_numpy() == 0), 1)),
        "Test_precision_at_validation_threshold": precision_score(y_test, test_at_operating, zero_division=0),
        "Test_F1_at_validation_threshold": f1_score(y_test, test_at_operating, zero_division=0),
        "Train_seconds": training_seconds,
        "Inference_ms_per_1000": (inference_seconds / max(len(y_test), 1)) * 1_000_000,
    }
    results.append(row)
    predictions_val[name] = val_probabilities
    predictions_test[name] = test_probabilities
    predictions[name] = test_probabilities
    model_store[name] = {"model": fitted_model, "cols": list(cols), "kind": kind}
    print(f"Finished {name}: validation ROC-AUC={row['Validation_ROC_AUC']:.4f}, test ROC-AUC={row['Test_ROC_AUC']:.4f}, test AP={row['Test_AP']:.4f}, train={training_seconds:.1f}s")

def fit_sklearn_model(name, estimator, cols, kind):
    t0 = time.perf_counter()
    estimator.fit(X_train[cols], y_train)
    train_s = time.perf_counter() - t0
    t1 = time.perf_counter()
    val_probabilities = estimator.predict_proba(X_val[cols])[:, 1]
    test_probabilities = estimator.predict_proba(X_test[cols])[:, 1]
    inference_s = time.perf_counter() - t1
    record_result(name, cols, kind, estimator, val_probabilities, test_probabilities, train_s, inference_s)

print("Split complete. Models fit on training rows; validation is used for operating-point selection; test rows remain held out.")


## 7. Classical AML baselines

We compare Logistic Regression (a transparent linear baseline) and a nonlinear gradient-boosted tree. Each is trained on low-level features, high-level features, and all 28 features. XGBoost is preferred; if its optional installation fails, the notebook uses scikit-learn's HistGradientBoostingClassifier as a fallback.

In [ ]:
FEATURE_SETS = {
    "Low-level (21)": LOW_LEVEL,
    "High-level (7)": HIGH_LEVEL,
    "All features (28)": FEATURE_NAMES,
}

for set_name, cols in FEATURE_SETS.items():
    lr = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=250, solver="lbfgs", random_state=SEED))
    ])
    fit_sklearn_model(f"Logistic Regression | {set_name}", lr, cols, set_name)

for set_name, cols in FEATURE_SETS.items():
    if HAS_XGBOOST:
        tree_model = XGBClassifier(
            n_estimators=120,
            max_depth=5,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED,
        )
        tree_name = "XGBoost"
    else:
        tree_model = HistGradientBoostingClassifier(
            max_iter=TREE_MAX_ITER,
            learning_rate=0.08,
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=SEED,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
        )
        tree_name = "HistGradientBoosting fallback"
    fit_sklearn_model(f"{tree_name} | {set_name}", tree_model, cols, set_name)

print(f"Classical model runs completed. Boosted-tree implementation: {'XGBoost' if HAS_XGBOOST else 'HistGradientBoosting fallback'}")


## 8. Deep learning model: compact MLP

The MLP uses two hidden layers, batch normalization, ReLU, and dropout. The low-level-only MLP tests whether learning from raw detector features works without the seven derived features; the full-feature MLP tests the same architecture with all inputs. Each scaler is fitted only on the training split.

In [ ]:
class HiggsMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x)

def infer_mlp_probabilities(model, x_array):
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(x_array)), batch_size=8192, shuffle=False)
    outputs = []
    with torch.no_grad():
        for (xb,) in loader:
            logits = model(xb.to(DEVICE))
            outputs.append(torch.sigmoid(logits).cpu().numpy().reshape(-1))
    return np.concatenate(outputs)

def fit_mlp(name, cols, kind):
    start = time.perf_counter()
    scaler = StandardScaler()
    xtr = scaler.fit_transform(X_train[cols]).astype(np.float32)
    xval = scaler.transform(X_val[cols]).astype(np.float32)
    xte = scaler.transform(X_test[cols]).astype(np.float32)
    ytr = y_train.to_numpy(dtype=np.float32).reshape(-1, 1)

    dataset = TensorDataset(torch.from_numpy(xtr), torch.from_numpy(ytr))
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

    model = HiggsMLP(len(cols)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(MLP_EPOCHS):
        model.train()
        running_loss = 0.0
        seen = 0
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)
            seen += len(xb)
        print(f"{name} | epoch {epoch + 1}/{MLP_EPOCHS} | loss={running_loss / max(seen, 1):.4f}")

    training_seconds = time.perf_counter() - start
    infer_start = time.perf_counter()
    val_probabilities = infer_mlp_probabilities(model, xval)
    test_probabilities = infer_mlp_probabilities(model, xte)
    inference_seconds = time.perf_counter() - infer_start

    fitted = {"network": model, "scaler": scaler}
    record_result(name, cols, kind, fitted, val_probabilities, test_probabilities, training_seconds, inference_seconds)

fit_mlp("MLP | Low-level (21)", LOW_LEVEL, "Low-level (21)")
fit_mlp("MLP | All features (28)", FEATURE_NAMES, "All features (28)")
print("MLP runs completed.")


## 9. Compare models without using the test set for model selection

The table is ordered by validation ROC-AUC for transparent candidate selection. Test-set metrics are reported only after the model configurations have been fixed. The threshold shown is selected on validation data to meet 80% signal efficiency; test metrics at that threshold measure how well the operating point transfers to held-out events.

In [ ]:
results_df = pd.DataFrame(results).sort_values("Validation_ROC_AUC", ascending=False).reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)
display(results_df.round(4))

selected_by_validation = results_df.iloc[0]
print(f"Candidate selected by validation ROC-AUC: {selected_by_validation['Model']} (validation ROC-AUC={selected_by_validation['Validation_ROC_AUC']:.4f})")

val_frame = pd.DataFrame({"label": y_val.to_numpy()})
for model_name, probs in predictions_val.items():
    val_frame[model_name] = probs
val_frame.to_csv(OUTPUT_DIR / "val_predictions.csv", index=False)

test_frame = pd.DataFrame({"label": y_test.to_numpy()})
for model_name, probs in predictions_test.items():
    test_frame[model_name] = probs
test_frame.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

metadata = {
    "project": "HiggsLens",
    "data_source": DATA_SOURCE,
    "rows_loaded": int(len(X)),
    "feature_count": int(X.shape[1]),
    "low_level_feature_count": int(len(LOW_LEVEL)),
    "expert_derived_feature_count": int(len(HIGH_LEVEL)),
    "label_map": {"0": "background", "1": "signal"},
    "seed": int(SEED),
    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},
    "threshold_selection": "validation threshold chosen to achieve at least 80% signal efficiency",
    "test_protocol": "stratified split of the loaded subset; not the official final 500,000-row UCI test protocol",
    "device": str(DEVICE),
    "boosted_tree": "XGBoost" if HAS_XGBOOST else "HistGradientBoosting fallback",
    "mlp_epochs": int(MLP_EPOCHS),
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved results.csv, val_predictions.csv, test_predictions.csv, and run_metadata.json.")


## 10. Evaluation plots

Curves and calibration diagnostics use the held-out test predictions for the pre-planned set of models. The list of plotted models and the illustrative calibration model are chosen using validation ranking, not test ranking. Calibration on this subset is a diagnostic, not proof that probabilities correspond to real-world Higgs probabilities.

In [ ]:
plot_names = results_df["Model"].head(min(5, len(results_df))).tolist()
fig, ax = plt.subplots(figsize=(7, 6))
for name in plot_names:
    fpr, tpr, _ = roc_curve(y_test, predictions_test[name])
    auc = roc_auc_score(y_test, predictions_test[name])
    ax.plot(fpr, tpr, label=f"{name} (test AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", label="Random ranking")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves - held-out test split")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "roc_curves.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name in plot_names:
    precision, recall, _ = precision_recall_curve(y_test, predictions_test[name])
    ap = average_precision_score(y_test, predictions_test[name])
    ax.plot(recall, precision, label=f"{name} (test AP={ap:.3f})")
ax.axhline(float(y_test.mean()), linestyle="--", label="Positive prevalence baseline")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-recall curves - held-out test split")
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pr_curves.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
plot_results = results_df.sort_values("Validation_ROC_AUC", ascending=True)
ax.barh(plot_results["Model"], plot_results["Test_ROC_AUC"])
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("Test ROC-AUC")
ax.set_title("Held-out test performance (candidates ranked using validation)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "model_comparison_roc_auc.png", dpi=160)
plt.show()

calibration_model_name = selected_by_validation["Model"]
frac_pos, mean_pred = calibration_curve(y_test, predictions_test[calibration_model_name], n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mean_pred, frac_pos, marker="o", label=calibration_model_name)
ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
ax.set(xlabel="Mean predicted probability", ylabel="Observed signal fraction", title="Calibration - selected by validation, assessed on test")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration_selected_model.png", dpi=160)
plt.show()

print(f"Saved figures and results to: {OUTPUT_DIR.resolve()}")


## 11. Explainability: permutation importance for the strongest classical candidate

The classical candidate is selected by validation ROC-AUC. Permutation importance is computed on a validation subset so the final test split is not used for feature selection. Importance is an association-based diagnostic, not a causal effect; correlated features can share or mask importance.

In [ ]:
classical_rows = results_df[~results_df["Model"].str.startswith("MLP")]
if len(classical_rows):
    best_classical_name = classical_rows.iloc[0]["Model"]
    bundle = model_store[best_classical_name]
    best_classical_model = bundle["model"]
    imp_cols = bundle["cols"]
    imp_n = min(5000, len(X_val))
    imp_rng = np.random.default_rng(SEED)
    imp_idx = imp_rng.choice(len(X_val), size=imp_n, replace=False)
    imp_X = X_val.iloc[imp_idx][imp_cols]
    imp_y = y_val.iloc[imp_idx]
    importance = permutation_importance(
        best_classical_model, imp_X, imp_y,
        scoring="roc_auc", n_repeats=3, random_state=SEED,
        n_jobs=1, max_samples=1.0
    )
    importance_df = pd.DataFrame({
        "feature": imp_cols,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }).sort_values("importance_mean", ascending=False)
    importance_df.to_csv(OUTPUT_DIR / "permutation_importance.csv", index=False)
    display(importance_df.head(12).round(5))

    top_imp = importance_df.head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top_imp["feature"], top_imp["importance_mean"], xerr=top_imp["importance_std"])
    ax.set_xlabel("Decrease in validation ROC-AUC after shuffling")
    ax.set_title(f"Top permutation importances - {best_classical_name}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "permutation_importance.png", dpi=160)
    plt.show()
else:
    print("No classical model available for permutation importance.")


## 12. Generate a factual run summary

The summary records the real data source and actual event count. It distinguishes validation-based candidate selection from test-set reporting and explicitly labels the subset protocol. Review the output and describe actual findings; the notebook has not embedded pre-filled performance values.

In [ ]:
table_text = results_df.round(4).to_string(index=False)
summary = f"""# HiggsLens — HIGGS Classification Run Summary

## Experiment configuration
- Data source: {DATA_SOURCE}
- Events actually loaded: {len(X):,}
- Features: 28 total (21 low-level, 7 expert-derived high-level)
- Split: 64% train / 16% validation / 20% test from this loaded subset; seed={SEED}
- Training device for MLP: {DEVICE}
- Boosted-tree implementation: {'XGBoost' if HAS_XGBOOST else 'HistGradientBoosting fallback'}
- MLP epochs: {MLP_EPOCHS}; fallback gradient-boosting max iterations: {TREE_MAX_ITER}
- Label convention: 1=signal, 0=background
- Threshold selection: validation-only; target at least 80% signal efficiency
- Test protocol: random stratified subset split; this is not the official final-500,000-row test protocol

## Model comparison
```text
{table_text}
```

## Selection and measured leaders
- Candidate selected by validation ROC-AUC: **{selected_by_validation['Model']}** (validation ROC-AUC {selected_by_validation['Validation_ROC_AUC']:.4f}).

## Questions to answer in the report
1. Did expert-derived features improve Logistic Regression or boosted trees relative to low-level-only inputs?
2. Did the MLP's test performance justify its measured compute cost?
3. At the threshold selected on validation to target 80% signal efficiency, what background rejection transferred to the test split?
4. Did the best ranking model also have the strongest calibration or best inference cost?
5. Which features mattered on validation, and how might correlations affect the interpretation?

## Limitations
- The dataset contains simulated Monte Carlo events; outputs are not evidence of a real Higgs discovery.
- A subset split is not equivalent to the original benchmark's final 500,000-row test protocol.
- Benchmark class prevalence is not the real-world Higgs event rate; probabilities should not be called real-world Higgs probabilities.
- Physical discovery sensitivity requires event weights, expected signal/background yields, cross sections, integrated luminosity, selections, and systematic uncertainties. This basic HIGGS table does not supply everything needed for a defensible physics significance estimate.
- A single seed and fixed small MLP training budget are appropriate for the MVP, not a definitive benchmark. The end-semester plan adds repeated seeds, scale tests, and uncertainty intervals.

## References
- UCI Machine Learning Repository, HIGGS, DOI: https://doi.org/10.24432/C5V312
- Baldi, Sadowski & Whiteson (2014), *Searching for exotic particles in high-energy physics with deep learning*, https://doi.org/10.1038/ncomms5308
- UCI HIGGS data directory and readme: https://mlphysics.ics.uci.edu/data/higgs/
"""
(OUTPUT_DIR / "summary.md").write_text(summary, encoding="utf-8")
print(summary)
print(f"\nSummary saved to: {(OUTPUT_DIR / 'summary.md').resolve()}")


## What the results do and do not mean

Read the outputs with three questions in mind:

1. Which model ranked signal events above background best on the held-out sample (ROC-AUC and average precision)?
2. At the threshold selected on validation data, how much background is rejected while retaining signal?
3. Is any improvement worth the observed time cost, and do the calibration results support treating the scores as probabilities within this simulated dataset?

A single test split gives an estimate, not certainty. The mid-semester report should call the results preliminary. The end-semester plan adds repeated seeds, uncertainty estimates and scale checks.


## 13. Export notebook outputs for HiggsLens

The output ZIP contains metrics, predictions, metadata, figures, and the summary—not the raw HIGGS dataset or trained model weights. The Streamlit app uses the prediction and metric artifacts to explore operating thresholds without retraining a model on every page view.

In [ ]:
import zipfile
zip_path = Path("higgs_project_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(OUTPUT_DIR.parent)))
print(f"Created: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:.1f} KB)")


## 14. Final submission checklist

- [ ] Run all cells top-to-bottom without manually editing metric results.
- [ ] Keep `results.csv`, `val_predictions.csv`, `test_predictions.csv`, `run_metadata.json`, `summary.md`, and all plots.
- [ ] State the actual source and row count printed by the notebook.
- [ ] Explain train/validation/test separation and that this subset split is not the official UCI test protocol.
- [ ] Include model configurations, ROC-AUC, average precision, calibration, selected-threshold metrics, runtime, and feature analysis.
- [ ] Do not tune thresholds on the test set or claim that a lower Brier score alone proves better calibration.
- [ ] Do not invent numbers or claim a real-world Higgs discovery.
- [ ] For the end-semester benchmark, use the official final 500,000 rows as a locked test set where compute permits; otherwise label the deviation explicitly.

### Suggested conclusion structure
“On the tested subset of **[actual N]** simulated events, **[candidate selected using validation]** achieved **[test ROC-AUC]** test ROC-AUC and **[test AP]** average precision. At a threshold selected on validation to target 80% signal efficiency, its test background rejection was **[measured value]**. Its training and inference costs were **[measured values]**. This supports **[actual conclusion]**, subject to **[limitations]**. This is a simulated-data classification benchmark, not a claim of real-world particle discovery.”